# Aula 07 — Testes de Hipótese (z e t)

## Objetivos
1. Estruturar um teste de hipótese: $H_0$ vs $H_1$.
2. Entender erros do tipo I e II, nível de significância $\alpha$ e poder.
3. Realizar **teste z** (variância conhecida) e **teste t** (variância desconhecida).
4. Interpretar **p-valor** corretamente.

---

## 1. Anatomia de um teste

1. **Formular hipóteses**:
   - $H_0$ (nula): status quo, "não há diferença".
   - $H_1$ (alternativa): o que queremos provar.

2. **Escolher o nível de significância** $\alpha$ (tipicamente 0,05).

3. **Calcular a estatística do teste** sob $H_0$.

4. **Decidir**: comparar p-valor com $\alpha$, ou estatística com valor crítico.

5. **Concluir** em termos do problema.

## 2. Tipos de erro

|                | $H_0$ verdadeira | $H_0$ falsa |
|----------------|------------------|-------------|
| Não rejeita $H_0$ | ✓ Decisão correta | Erro tipo II ($\beta$) |
| Rejeita $H_0$     | Erro tipo I ($\alpha$) | ✓ Decisão correta (poder = $1-\beta$) |

## 3. Estatísticas dos testes

### Teste z (σ conhecido)
$$z = \frac{\bar{x} - \mu_0}{\sigma/\sqrt{n}}$$

### Teste t (σ desconhecido)
$$t = \frac{\bar{x} - \mu_0}{s/\sqrt{n}} \sim t_{n-1}$$

## 4. Hipóteses uni/bilaterais

| $H_1$ | Tipo | Região crítica |
|-------|------|----------------|
| $\mu \ne \mu_0$ | bilateral | duas caudas |
| $\mu > \mu_0$   | unilateral à direita | cauda direita |
| $\mu < \mu_0$   | unilateral à esquerda | cauda esquerda |

## 5. Interpretação do p-valor

**p-valor:** probabilidade, sob $H_0$, de observar uma estatística tão ou mais extrema
que a observada.

- $p < \alpha$ → **rejeita** $H_0$.
- $p \ge \alpha$ → **não rejeita** $H_0$.

> ⚠️ p-valor NÃO é "a probabilidade de $H_0$ ser verdadeira".

In [ ]:
# === SETUP PARA GOOGLE COLAB ===
# Esta célula garante que o notebook funcione tanto em ambiente local
# quanto em quem clonar o repositório direto no Colab.

import sys, os, subprocess

# 1) Instala dependências (silencioso se já existirem)
pkgs = ["numpy", "pandas", "matplotlib", "seaborn", "scipy", "requests", "plotly", "statsmodels"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# 2) Se estivermos no Colab e o repo ainda não foi clonado, clona.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/220719/curso-estatistica-python.git"  # ajuste após criar o repo

if IN_COLAB and not os.path.exists("/content/curso-estatistica-python"):
    subprocess.run(["git", "clone", REPO_URL, "/content/curso-estatistica-python"], check=False)

# 3) Adiciona o diretório utils ao PYTHONPATH para importar o módulo SIDRA
BASE = "/content/curso-estatistica-python" if IN_COLAB else ".."
if BASE not in sys.path:
    sys.path.append(BASE)

# 4) Pasta de gráficos (cria se não existir)
GRAFICOS_DIR = os.path.join(BASE, "graficos")
os.makedirs(GRAFICOS_DIR, exist_ok=True)

print("✓ Ambiente pronto. Pasta de gráficos:", GRAFICOS_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

sns.set_theme(style="whitegrid")

from utils.sidra_api import obter_ipca_mensal

## 6. Aplicação: a meta de inflação foi atingida?

O Banco Central do Brasil adota a meta de **3% ao ano** (≈ 0,2466% ao mês, geometricamente).

Pergunta: **a média do IPCA dos últimos 24 meses é compatível com a meta?**

- $H_0$: $\mu = 0{,}2466$
- $H_1$: $\mu \ne 0{,}2466$  (teste bilateral)
- $\alpha = 0{,}05$

In [ ]:
df = obter_ipca_mensal(24)
x = df["variacao_mensal"].values
n = len(x)
media = x.mean()
s = x.std(ddof=1)

mu0 = (1.03)**(1/12) - 1   # converte 3% a.a. em mensal equivalente
mu0_pct = mu0 * 100

print(f"Meta mensal equivalente a 3% a.a.: {mu0_pct:.4f}%")
print(f"Média observada (24 meses):       {media:.4f}%")
print(f"Desvio-padrão amostral:           {s:.4f}")
print(f"n = {n}")

In [ ]:
# Teste t bilateral — passo a passo
t_stat = (media - mu0_pct) / (s / np.sqrt(n))
gl = n - 1

# p-valor bilateral
p_valor = 2 * (1 - stats.t.cdf(abs(t_stat), df=gl))

print(f"Estatística t: {t_stat:.4f}")
print(f"Graus de liberdade: {gl}")
print(f"p-valor: {p_valor:.4f}")

alpha = 0.05
t_critico = stats.t.ppf(1 - alpha/2, df=gl)
print(f"\nt crítico (α=0.05, bilateral): ±{t_critico:.4f}")

if p_valor < alpha:
    print(f"\nDecisão: REJEITA H0 (p < {alpha})")
    print("Conclusão: a média do IPCA é estatisticamente diferente da meta de 3% a.a.")
else:
    print(f"\nDecisão: NÃO REJEITA H0 (p ≥ {alpha})")
    print("Conclusão: não há evidência de que a média difira da meta.")

### Atalho com `scipy`

In [ ]:
# ttest_1samp faz exatamente o mesmo cálculo
res = stats.ttest_1samp(x, popmean=mu0_pct)
print(f"t = {res.statistic:.4f}, p = {res.pvalue:.4f}")

## 7. Visualizando a região crítica

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
tt = np.linspace(-5, 5, 400)
ax.plot(tt, stats.t.pdf(tt, gl), color="navy", linewidth=2, label=f"t({gl})")

# Regiões críticas
mask_l = tt < -t_critico
mask_r = tt >  t_critico
ax.fill_between(tt[mask_l], stats.t.pdf(tt[mask_l], gl), color="red", alpha=0.4, label=f"Região crítica α=0.05")
ax.fill_between(tt[mask_r], stats.t.pdf(tt[mask_r], gl), color="red", alpha=0.4)

# Estatística observada
ax.axvline(t_stat, color="green", linewidth=2, linestyle="--",
           label=f"t obs = {t_stat:.2f}")
ax.set_title("Teste t bilateral — IPCA vs meta de 3% a.a.", fontweight="bold")
ax.set_xlabel("t")
ax.set_ylabel("Densidade")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, "aula07_teste_t.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Teste unilateral

E se quisermos testar especificamente **se a inflação está ACIMA da meta**?

- $H_0$: $\mu \le \mu_0$
- $H_1$: $\mu > \mu_0$

In [ ]:
# p-valor unilateral à direita
p_uni = 1 - stats.t.cdf(t_stat, df=gl)
print(f"t = {t_stat:.4f}")
print(f"p-valor unilateral (cauda direita): {p_uni:.4f}")
print()
if p_uni < 0.05:
    print("REJEITA H0: evidência de que o IPCA médio está ACIMA da meta.")
else:
    print("Não rejeita H0.")

## 9. Poder do teste — quanto $n$ preciso?

O poder é $1 - \beta$ = probabilidade de **detectar** uma diferença real.

Vamos plotar a curva de poder em função de $n$ para detectar diferença de 0,1 ponto percentual.

In [ ]:
from scipy.stats import nct  # t não-central

def poder_teste_t(n, delta, sigma, alpha=0.05):
    """Poder de um teste t bilateral para detectar diferença 'delta'."""
    gl = n - 1
    nc = delta * np.sqrt(n) / sigma     # parâmetro de não-centralidade
    t_c = stats.t.ppf(1 - alpha/2, df=gl)
    # P(|T| > t_c | H1) = 1 - F_nct(t_c) + F_nct(-t_c)
    return 1 - nct.cdf(t_c, gl, nc) + nct.cdf(-t_c, gl, nc)

ns = np.arange(5, 201, 5)
delta_pp = 0.10  # diferença a detectar: 0.1 ponto percentual
poderes = [poder_teste_t(nn, delta_pp, s) for nn in ns]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ns, poderes, color="darkorange", linewidth=2)
ax.axhline(0.8, color="red", linestyle="--", label="Poder = 0.80")
ax.set_xlabel("Tamanho da amostra (n)")
ax.set_ylabel("Poder")
ax.set_title(f"Curva de poder — detectar diferença de {delta_pp} p.p. (σ = {s:.2f})",
             fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, "aula07_poder.png"), dpi=150, bbox_inches="tight")
plt.show()

## Exercícios

1. Teste se a média do IPCA dos **últimos 60 meses** difere de 0,4% (bilateral, α=0,05).
2. Refaça o teste anterior em modo unilateral à direita.
3. Calcule o tamanho de amostra necessário para detectar diferença de 0,05 p.p. com
   poder de 90%.

**Próxima aula:** Testes para duas amostras + qui-quadrado.